In [1]:
# 1) Install / upgrade required libraries
!pip install -q pandas scikit-learn matplotlib torch torchvision timm


In [2]:
# 2) Imports
import os
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import timm  # for Vision Transformer variants

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split  # if you want to rerun splits


In [3]:
# 3) Set device to GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [4]:
# 4) Load CSVs
train_csv     = "splits/train.csv"
val_csv       = "splits/val.csv"
test_csv      = "splits/test.csv"
label2id_csv  = "splits/label2id.csv"

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)
test_df  = pd.read_csv(test_csv)

label2id_df = pd.read_csv(label2id_csv)
label2id    = dict(zip(label2id_df["country"], label2id_df["id"]))
id2label    = {v: k for k, v in label2id.items()}

num_classes = len(label2id)
print(f"Train samples: {len(train_df):,}, Val samples: {len(val_df):,}, Test samples: {len(test_df):,}")
print(f"Number of classes: {num_classes}")


Train samples: 40,000, Val samples: 4,998, Test samples: 4,999
Number of classes: 124


In [5]:
# 5) Image transforms
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
])

val_test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
])

# 6) Custom Dataset
class GeoGuessrDataset(Dataset):
    def __init__(self, dataframe, label_map, transform):
        """
        dataframe: pd.DataFrame with ["file_path","label"]
        label_map:  dict mapping country→id
        transform:  torchvision.transforms for images
        """
        self.df = dataframe.reset_index(drop=True)
        self.label_map = label_map
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["file_path"]).convert("RGB")
        img = self.transform(img)
        label = self.label_map[row["label"]]
        return img, label

# 7) Instantiate Datasets
train_dataset = GeoGuessrDataset(train_df, label2id, train_transforms)
val_dataset   = GeoGuessrDataset(val_df,   label2id, val_test_transforms)
test_dataset  = GeoGuessrDataset(test_df,  label2id, val_test_transforms)


In [6]:
# 8) Create DataLoaders
batch_size   = 32
num_workers  = 2  # set 0 if you run into issues

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=0, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=0, pin_memory=True
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


Train batches: 1250, Val batches: 157, Test batches: 157


In [7]:
# 9) ResNet-50 setup
resnet = models.resnet50(pretrained=True)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet = resnet.to(device)

# 10) ViT-B_16 setup via timm ("vit_base_patch16_224" is a commonly used variant)
vit = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=num_classes)
vit = vit.to(device)

print("ResNet-50 and ViT-B_16 initialized:")
print("  - ResNet-50 head:", resnet.fc)
print("  - ViT head:      ", vit.head)  # timm’s ViT uses attribute `vit.head` for classification layer


/opt/miniconda3/envs/cs224n/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/miniconda3/envs/cs224n/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet-50 and ViT-B_16 initialized:
  - ResNet-50 head: Linear(in_features=2048, out_features=124, bias=True)
  - ViT head:       Linear(in_features=768, out_features=124, bias=True)


In [ ]:
from torch.nn.functional import softmax, log_softmax

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction
    def forward(self, logits, targets):
        logp = log_softmax(logits, dim=1)           # log-prob
        p     = logp.exp()                          # prob
        focal = (1 - p).pow(self.gamma) * logp
        loss  = nn.NLLLoss(weight=self.weight, reduction=self.reduction)(focal, targets)
        return -loss

In [8]:
# 11) Common criterion + optimizers
criterion = nn.CrossEntropyLoss()

resnet_optimizer = torch.optim.AdamW(resnet.parameters(), lr=3e-5, weight_decay=1e-3)
resnet_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    resnet_optimizer, mode="max", factor=0.5, patience=1
)

vit_optimizer = torch.optim.AdamW(vit.parameters(), lr=3e-5, weight_decay=1e-3)
vit_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    vit_optimizer, mode="max", factor=0.5, patience=1,
)


In [9]:
def train_validate_one_epoch(model, train_loader, val_loader,
                             criterion, optimizer, device):
    """
    Runs one epoch of training + validation on `model`.
    Returns: train_loss_avg, train_acc, val_loss_avg, val_acc
    """
    # ——— Training ———
    model.train()
    running_loss = 0.0
    total_samples = 0
    correct_preds = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct_preds += (preds == labels).sum().item()
        total_samples += images.size(0)

    train_loss_avg = running_loss / total_samples
    train_acc = correct_preds / total_samples

    # ——— Validation ———
    model.eval()
    val_running_loss = 0.0
    val_total = 0
    val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss_avg = val_running_loss / val_total
    val_acc = val_correct / val_total

    return train_loss_avg, train_acc, val_loss_avg, val_acc


In [10]:
# 12) Training loop for ResNet-50
num_epochs = 1
best_resnet_val_acc = 0.0
resnet_save_path = "resnet50_geoguessr_best.pth"

resnet_history = []  # to store (train_loss, train_acc, val_loss, val_acc) per epoch

for epoch in range(num_epochs):
    train_l, train_a, val_l, val_a = train_validate_one_epoch(
        resnet, train_loader, val_loader,
        criterion, resnet_optimizer, device
    )
    resnet_scheduler.step(val_a)

    resnet_history.append((train_l, train_a, val_l, val_a))

    print(
        f"[ResNet] Epoch {epoch+1}/{num_epochs}  "
        f"Train Loss: {train_l:.4f}, Train Acc: {train_a:.4f}  "
        f"Val Loss: {val_l:.4f}, Val Acc: {val_a:.4f}"
    )

    if val_a > best_resnet_val_acc:
        best_resnet_val_acc = val_a
        torch.save(resnet.state_dict(), resnet_save_path)
        print(f"  → New best ResNet saved (val_acc={val_a:.4f})\n")
    else:
        print()

print(f"Best ResNet validation accuracy: {best_resnet_val_acc:.4f}")


/opt/miniconda3/envs/cs224n/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 

In [13]:
# ── SAMPLE 50 IMAGES FOR A QUICK TEST ──

# Assume train_df is your full DataFrame of (file_path, label).
# We’ll randomly pick 50 rows (seeded for reproducibility).
small_train_df = train_df.sample(n=5, random_state=42).reset_index(drop=True)

# Now build a tiny Dataset and DataLoader from those 50.
small_train_dataset = GeoGuessrDataset(small_train_df, label2id, train_transforms)

# Use num_workers=0 so we don’t hit any pickling issues.
small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=16,      # you can choose any batch_size ≤50
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

# (Optionally) print to confirm
print(f"Using {len(small_train_dataset)} training images in this test run.")
print(f"→ {len(small_train_loader)} batches (batch_size=16 × {len(small_train_loader)} ≈ 50 images)")


Using 5 training images in this test run.
→ 1 batches (batch_size=16 × 1 ≈ 50 images)


In [14]:
# 12) Training loop on 50 images
num_epochs = 1
best_resnet_val_acc = 0.0
resnet_save_path = "resnet50_geoguessr_best.pth"

resnet_history = []

for epoch in range(num_epochs):
    # Notice we pass small_train_loader here instead of train_loader
    train_l, train_a, val_l, val_a = train_validate_one_epoch(
        resnet,
        small_train_loader,  # ← only 50 images
        val_loader,          # keep full validation set if you like, or similarly subsample it
        criterion,
        resnet_optimizer,
        device
    )
    resnet_scheduler.step(val_a)

    resnet_history.append((train_l, train_a, val_l, val_a))

    print(
        f"[ResNet‐50 on 50 imgs] Epoch {epoch+1}/{num_epochs}  "
        f"Train Loss: {train_l:.4f}, Train Acc: {train_a:.4f}  "
        f"Val Loss: {val_l:.4f}, Val Acc: {val_a:.4f}"
    )

    if val_a > best_resnet_val_acc:
        best_resnet_val_acc = val_a
        torch.save(resnet.state_dict(), resnet_save_path)
        print(f"  → New best ResNet saved (val_acc={val_a:.4f})\n")
    else:
        print()

print(f"Best ResNet validation accuracy (on this 50‐image run): {best_resnet_val_acc:.4f}")


[ResNet‐50 on 50 imgs] Epoch 1/1  Train Loss: 4.5275, Train Acc: 0.0000  Val Loss: 4.4946, Val Acc: 0.0728
  → New best ResNet saved (val_acc=0.0728)

Best ResNet validation accuracy (on this 50‐image run): 0.0728


In [ ]:
# 13) Training loop for ViT-B_16
best_vit_val_acc = 0.0
vit_save_path = "vit_b16_geoguessr_best.pth"

def evaluate_vit_topk(model, weight_path, loader, device, topk=(1,5)):
    """
    Returns a dict with:
      - 'top1_acc'
      - 'top5_acc'
      - 'y_true'   (list of ground‐truth indices)
      - 'y_pred1'  (list of top-1 predictions)
      - 'y_pred5'  (list of lists of top-5 predicted indices)
    """
    # Load best weights, move to eval
    model.load_state_dict(torch.load(weight_path, map_location=device))
    model.to(device).eval()

    total_samples = 0
    correct_top1 = 0
    correct_top5 = 0

    all_true = []
    all_top1 = []
    all_top5 = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)  # shape [batch_size, num_classes]

            # Top-1
            _, pred1 = outputs.topk(k=1, dim=1, largest=True, sorted=True)
            pred1 = pred1.view(-1)  # shape [batch_size]
            correct_top1 += (pred1 == labels).sum().item()

            # Top-5
            _, pred5 = outputs.topk(k=5, dim=1, largest=True, sorted=True)
            # pred5 shape: [batch_size, 5]
            # compare each label to the 5 predictions
            # For each i, check if labels[i] is in pred5[i,:]
            # We can do a vectorized check:
            #   labels.unsqueeze(1) has shape [batch_size, 1]
            #   pred5 has shape [batch_size, 5]
            #   eq_mat has shape [batch_size, 5] of bool
            eq_mat = pred5.eq(labels.view(-1,1))
            # If any of the 5 is True, then that sample is correct in Top-5
            correct_top5 += eq_mat.any(dim=1).sum().item()

            batch_size = labels.size(0)
            total_samples += batch_size

            all_true.extend(labels.cpu().tolist())
            all_top1.extend(pred1.cpu().tolist())
            all_top5.extend(pred5.cpu().tolist())  # each entry is a list of 5

    top1_acc = correct_top1 / total_samples
    top5_acc = correct_top5 / total_samples

    return {
        "top1_acc": top1_acc,
        "top5_acc": top5_acc,
        "y_true": all_true,
        "y_pred1": all_top1,
        "y_pred5": all_top5,
    }

vit_results = evaluate_vit_topk(
    vit, vit_save_path, test_loader, device, topk=(1,5)
)
print(f"ViT-B_16 Test Accuracy (Top-1): {vit_results['top1_acc']:.4f}")
print(f"ViT-B_16 Test Accuracy (Top-5): {vit_results['top5_acc']:.4f}")

/opt/miniconda3/envs/cs224n/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[ViT-B_16] Epoch 1/1  Train Loss: 5.5373, Train Acc: 0.0000  Val Loss: 4.9707, Val Acc: 0.0060
  → New best ViT saved (val_acc=0.0060)

Best ViT validation accuracy: 0.0060
